# Linear algebra and differential calculus

This notebook accompanies Exercises 1.1, 1.2 and 1.3 of the course. First establish the results mathematically; JAX and NumPy computations then check the formulae on examples.

We shall use, in particular, the Hadamard product, singular value decomposition (SVD), diagonal matrices and Jacobian-vector products (JVPs).


## Exercises

1. [Exercise 1.1 — Hadamard product](#exercice-1-1)
2. [Exercise 1.2 — Image compression using SVD](#exercice-1-2)
3. [Exercise 1.3 — Componentwise derivatives](#exercice-1-3)


In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_sample_image

print(f"JAX {jax.__version__}")


<a id="exercice-1-1"></a>
## Exercise 1.1 — Hadamard product

For $x,y\in\mathbb R^n$, the Hadamard product is

$$
x\odot y=(x_i y_i)_{1\leq i\leq n}.
$$

For $x,y,z\in\mathbb R^n$, show that

$$
x\odot y=\operatorname{Diag}(x)y=\operatorname{Diag}(y)x,
$$

$$
\operatorname{Diag}(x\odot y)
=\operatorname{Diag}(x)\operatorname{Diag}(y),
$$

$$
\langle x\odot y,z\rangle
=\langle x,y\odot z\rangle
=\langle y,x\odot z\rangle,
$$

and

$$
|x\odot y|\leq |x|\,|y|.
$$

### Verification with JAX

In JAX, the `*` operator is the Hadamard product for arrays of the same shape, `jnp.diag(x)` constructs $\operatorname{Diag}(x)$ and `jnp.linalg.norm(x)` computes the Euclidean norm.

Construct three random vectors and verify all four results. For floating-point equalities, use `jnp.allclose` rather than `==`.

In [ ]:
cle = jax.random.key(2026)
cle_x, cle_y, cle_z = jax.random.split(cle, 3)
n = 7
x = jax.random.normal(cle_x, (n,))
y = jax.random.normal(cle_y, (n,))
z = jax.random.normal(cle_z, (n,))

In [ ]:
# Complete this section.

> A numerical check can expose an incorrect formula, but cannot prove a general formula. Here, the proof relies on componentwise calculations.

<a id="exercice-1-2"></a>
## Exercise 1.2 — Image compression using SVD

A greyscale image is represented by a matrix $A\in\mathbb R^{m\times n}$. If

$$
A=U\Sigma V^{\mathsf T},
$$

its rank-$k$ truncated SVD is

$$
A_k=U_{:,1:k}\Sigma_{1:k,1:k}V_{:,1:k}^{\mathsf T}.
$$

1. Express the errors $\|A-A_k\|_2$ and $\|A-A_k\|_F$ in terms of the singular values.
2. Compare the $mn$ image coefficients with the $k(m+n+1)$ coefficients of the truncated SVD.
3. Plot the singular values and several approximations $A_k$.
4. Plot the relative Frobenius error and compression ratio against $k$.


### Image and matrix

The image `china.jpg` is supplied with scikit-learn: no download is required. We convert it to greyscale, subsample its pixels and scale its intensities to $[0,1]$.


In [ ]:
image_couleur = load_sample_image("china.jpg")[::2, ::2]
A = np.tensordot(
    image_couleur[..., :3], np.array([0.2126, 0.7152, 0.0722]), axes=1
) / 255.0
m, n = A.shape

fig, axe = plt.subplots(figsize=(6, 4))
axe.imshow(A, cmap="gray", vmin=0.0, vmax=1.0)
axe.set(title=f"original image: {m} × {n}")
axe.axis("off")
plt.show()


Compute the SVD with `np.linalg.svd(A, full_matrices=False)`. Plot the singular values, then reconstruct the images of rank 5, 20, 50 and 100.


In [ ]:
# Complete this section.

Verify the error formula for $k=20$. For all ranks, plot

$$
\frac{\|A-A_k\|_F}{\|A\|_F}
\quad\text{and}\quad
\frac{k(m+n+1)}{mn}.
$$


In [ ]:
# Complete this section.

This experiment concerns the low-rank factorisation of a particular matrix. In Chapter 2, we shall instead seek a single reconstruction machine for a whole family of data.


<a id="exercice-1-3"></a>
## Exercise 1.3 — Componentwise derivatives

Let $f\in C^2(\mathbb R,\mathbb R)$ and consider its componentwise extension to $\mathbb R^n$. Show that, for $x,h,k\in\mathbb R^n$,

$$
Df(x)h=f'(x)\odot h
=\operatorname{Diag}(f'(x))h
$$

and

$$
D^2f(x)[h,k]=f''(x)\odot h\odot k.
$$


### Verification with JAX

We choose
$$
f(t)=\sin(t)+\frac{t^3}{10}.
$$

The command `jax.jvp(F, (x,), (h,))` returns the pair $(F(x),DF(x)h)$. Applying a second JVP to $x\mapsto DF(x)h$ in direction $k$ directly gives $D^2F(x)[h,k]$, without constructing the full second-derivative tensor.

In [ ]:
def f(t):
    return jnp.sin(t) + 0.1 * t**3

def derivee_f(t):
    return jnp.cos(t) + 0.3 * t**2

def derivee_seconde_f(t):
    return -jnp.sin(t) + 0.6 * t

x = jnp.array([-1.0, -0.2, 0.4, 1.3])
h = jnp.array([0.5, -1.0, 2.0, 0.3])
k = jnp.array([-0.7, 0.4, 1.0, -2.0])

In [ ]:
# Complete this section.

### Further question

Compute `jax.jacfwd(f)(x)` and verify that the resulting matrix is diagonal with diagonal $f'(x)$. Why would this structure disappear for a function whose components mix several components of $x$?

In [ ]:
# Complete this section.

## Summary

The product `*` between arrays of the same shape represents the Hadamard product. The SVD describes a matrix using orthogonal directions ordered by importance; truncating it gives the best low-rank approximation. For a function applied componentwise, the Jacobian matrix is diagonal and directional derivatives are also computed componentwise.

The `jax.jvp` transformation directly gives the action of a derivative on a direction. It will be particularly useful for checking the action of a Hessian without explicitly forming a large matrix.
